In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
import mne
import pandas as pd
from pathlib import Path
from typing import Union, Dict, Any

def extract_edf_metadata(data_dir: Union[str, Path]) -> pd.DataFrame:
    """
    Iterates through a directory of EDF files and extracts core EEG metadata without loading data into memory.
    
    Args:
        data_dir (Union[str, Path]): Path to the directory containing .edf files.
        
    Returns:
        pd.DataFrame: A dataframe containing metadata for each file.
    
    Raises:
        ValueError: If the directory does not exist or contains no EDF files.
    """
    data_path = Path(data_dir)
    if not data_path.is_dir():
        raise ValueError(f"Directory not found: {data_path}")
        
    edf_files = list(data_path.rglob("*.edf"))
    if not edf_files:
        raise ValueError(f"No .edf files found in {data_path}")

    metadata_records = []
    
    # Suppress verbose MNE outputs during batch processing
    mne.set_log_level('WARNING')

    for file_path in edf_files:
        try:
            # preload=False is critical to avoid RAM exhaustion
            raw = mne.io.read_raw_edf(file_path, preload=False)
            
            info: Dict[str, Any] = {
                "file_name": file_path.name,
                "n_channels": len(raw.ch_names),
                "sampling_freq_hz": raw.info['sfreq'],
                "duration_sec": raw.n_times / raw.info['sfreq'],
                "highpass_hz": raw.info['highpass'],
                "lowpass_hz": raw.info['lowpass'],
                "channel_names": raw.ch_names
            }
            metadata_records.append(info)
            
        except Exception as e:
            print(f"[ERROR] Failed to read {file_path.name}: {e}")
            metadata_records.append({
                "file_name": file_path.name,
                "n_channels": None,
                "sampling_freq_hz": None,
                "duration_sec": None,
                "highpass_hz": None,
                "lowpass_hz": None,
                "channel_names": f"ERROR: {e}"
            })

    df_metadata = pd.DataFrame(metadata_records)
    return df_metadata

# ==========================================
# Minimal Usage Example
# ==========================================
if __name__ == "__main__":
    # FIX 1: Point to the parent directory, NOT the individual .edf file
    dataset_directory = "/kaggle/input/datasets/mridulmondal/normal01"
    
    # FIX 2: Removed Path().mkdir() - /kaggle/input/ is a read-only filesystem.
    # The data already exists there; no directory creation is necessary or allowed.
    
    try:
        # Pass the directory path to the extraction function
        metadata_df = extract_edf_metadata(dataset_directory)
        print(metadata_df.to_string(index=False))
        
        # Validation checks
        if not metadata_df.empty:
            unique_sfreqs = metadata_df['sampling_freq_hz'].dropna().unique()
            if len(unique_sfreqs) > 1:
                print(f"\n[WARNING] Inconsistent sampling frequencies detected: {unique_sfreqs}")
    except ValueError as e:
        print(f"[PIPELINE ERROR] {e}")

                                     file_name  n_channels  sampling_freq_hz  duration_sec  highpass_hz  lowpass_hz                                                                                                                            channel_names
Raw_Signal_Psycophysiological_Insomnia_010.edf          24             256.0       28774.0          0.0       128.0 [EOG1, EOG2, EOG1A1, EOG2A1, C4A1, C3A2, F3, F4, C3, C4, A1, A2, O1, O2, ECGII, EMG, EMG1, EMG2, EOG1A2, EOG2A2, F3A2, F4A1, O1A2, O2A1]
                         Normal_Subject_07.edf          24             256.0       28782.0          0.0       128.0 [EOG1, EOG2, EOG1A1, EOG2A1, C4A1, C3A2, F3, F4, C3, C4, A1, A2, O1, O2, ECGII, EMG, EMG1, EMG2, EOG1A2, EOG2A2, F3A2, F4A1, O1A2, O2A1]
 Raw_Signal_Psycophysiological_Insomnia_01.edf          24             256.0       28782.0          0.0       128.0 [EOG1, EOG2, EOG1A1, EOG2A1, C4A1, C3A2, F3, F4, C3, C4, A1, A2, O1, O2, ECGII, EMG, EMG1, EMG2, EOG1A2, EOG2A2, F3A2, F4A1, 

In [1]:
import os
import pandas as pd

def extract_txt_start_time(txt_path: str) -> str:
    """Reads the text file header and extracts the Start Time string."""
    if not os.path.exists(txt_path):
        return "File Not Found"
        
    try:
        # latin-1 encoding prevents crashes from medical unit symbols (e.g., ÁV)
        with open(txt_path, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if i > 10: break 
                if "Start Time:" in line:
                    return line.split("Start Time:")[1].strip()
    except Exception as e:
        return f"Error: {e}"
    return "No Start Time"

def extract_all_psg_times(txt_base_dir: str) -> pd.DataFrame:
    """Scans all subject folders and extracts the start times."""
    results = []
    
    # Get all subject folders
    if not os.path.exists(txt_base_dir):
        raise FileNotFoundError(f"Directory not found: {txt_base_dir}")
        
    folders = [f for f in os.listdir(txt_base_dir) if os.path.isdir(os.path.join(txt_base_dir, f))]
    
    for folder in sorted(folders):
        folder_path = os.path.join(txt_base_dir, folder)
        
        profile_path = os.path.join(folder_path, "Sleep Profile.txt")
        arousal_path = os.path.join(folder_path, "Arousal.txt")
        
        profile_time = extract_txt_start_time(profile_path)
        arousal_time = extract_txt_start_time(arousal_path)
        
        results.append({
            "Subject_Folder": folder,
            "Sleep_Profile_Start": profile_time,
            "Arousal_Start": arousal_time
        })

    return pd.DataFrame(results)

# ==========================================
# USAGE
# ==========================================
# Set this to the path where your folders are located
TXT_DIRECTORY = "/kaggle/input/datasets/johanliebert28/ensominia-psg-outputs" 

df_times = extract_all_psg_times(TXT_DIRECTORY)
print(df_times.to_string(index=False))

                           Subject_Folder    Sleep_Profile_Start          Arousal_Start
             PSG_Output_Normal_Subject_01  1/26/2015 10:56:00 PM  1/26/2015 10:56:24 PM
             PSG_Output_Normal_Subject_02  9/24/2014 11:09:00 PM  9/24/2014 11:09:13 PM
             PSG_Output_Normal_Subject_03 12/21/2015 11:11:00 PM 12/21/2015 11:11:02 PM
             PSG_Output_Normal_Subject_04  9/29/2015 10:46:30 PM  9/29/2015 10:46:38 PM
             PSG_Output_Normal_Subject_05   4/8/2016 11:53:30 PM   4/8/2016 11:53:35 PM
             PSG_Output_Normal_Subject_06  8/18/2015 11:15:00 PM  8/18/2015 11:15:26 PM
             PSG_Output_Normal_Subject_07   8/4/2015 11:12:00 PM   8/4/2015 11:12:17 PM
             PSG_Output_Normal_Subject_08  7/16/2014 11:21:30 PM  7/16/2014 11:21:46 PM
             PSG_Output_Normal_Subject_09 12/28/2013 11:23:00 PM 12/28/2013 11:23:14 PM
             PSG_Output_Normal_Subject_10  5/14/2015 11:16:30 PM  5/14/2015 11:16:46 PM
             PSG_Output_Normal_S

In [4]:
import os
import mne
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple

def get_absolute_anchor(txt_path: str, edf_time: datetime) -> float:
    """Calculates offset in seconds between EDF hardware clock and TXT clock."""
    try:
        with open(txt_path, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if i > 10: break
                if "Start Time:" in line:
                    t_str = line.split("Start Time:")[1].strip()
                    # Handle malformed Insomnia_02 string
                    if len(t_str) < 12: return 0.0 
                    txt_time = datetime.strptime(t_str, "%m/%d/%Y %I:%M:%S %p")
                    return (txt_time - edf_time).total_seconds()
    except Exception:
        pass
    return 0.0 # Fallback to EDF zero-point

def parse_relative_sec(time_str: str) -> float:
    """Converts HH:MM:SS,ms to seconds from 00:00:00."""
    t = datetime.strptime(time_str.strip(), "%H:%M:%S,%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def build_qc_masks(rem_path: str, rel_path: str, edf_time: datetime, max_time: float) -> List[Tuple[float, float]]:
    """Builds a list of (start_sec, end_sec) exclusion zones for REM and Low Reliability."""
    exclusion_zones = []
    
    # 1. Parse REM
    rem_offset = get_absolute_anchor(rem_path, edf_time)
    try:
        with open(rem_path, 'r') as f:
            for line in f:
                if '-' in line and 'REM' in line:
                    t_range = line.split(';')[0]
                    start = parse_relative_sec(t_range.split('-')[0]) + rem_offset
                    end = parse_relative_sec(t_range.split('-')[1]) + rem_offset
                    exclusion_zones.append((start, end))
    except FileNotFoundError: pass

    # 2. Parse Reliability (<35%)
    rel_offset = get_absolute_anchor(rel_path, edf_time)
    try:
        with open(rel_path, 'r') as f:
            for line in f:
                if '<35% Reliability' in line:
                    t_str = line.split(';')[0]
                    start = parse_relative_sec(t_str) + rel_offset
                    exclusion_zones.append((start, start + 30.0)) # 30s epoch
    except FileNotFoundError: pass

    return exclusion_zones

def is_valid_slice(start: float, end: float, exclusion_zones: List[Tuple[float, float]]) -> bool:
    """Checks if a requested slice overlaps with any QC exclusion zone."""
    for ex_start, ex_end in exclusion_zones:
        if max(start, ex_start) < min(end, ex_end):
            return False
    return True

def clean_and_reconstruct(edf_path: str) -> Tuple[mne.io.RawArray, datetime]:
    """Executes Steps 1-3: Base ingestion, ICA, and 17-channel derivation."""
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    edf_time = raw.info['meas_date'].replace(tzinfo=None)
    sfreq = raw.info['sfreq']
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    eog_chans = ['EOG1', 'EOG2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    raw_base = raw.copy().pick_channels(base_eeg + eog_chans + emg_chans)
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    # EOG-Anchored ICA
    ica = mne.preprocessing.ICA(n_components=10, method='picard', random_state=42)
    ica.fit(raw_base.copy().pick_channels(base_eeg + eog_chans), verbose='ERROR')
    eog_idx, _ = ica.find_bads_eog(raw_base, ch_name=eog_chans, verbose='ERROR')
    ica.exclude = eog_idx
    raw_clean = ica.apply(raw_base, verbose='ERROR')
    
    data = raw_clean.get_data()
    idx = {ch: i for i, ch in enumerate(raw_clean.ch_names)}
    
    # Mathematical Reconstruction
    c4a1, c3a2 = data[idx['C4']] - data[idx['A1']], data[idx['C3']] - data[idx['A2']]
    f3a2, f4a1 = data[idx['F3']] - data[idx['A2']], data[idx['F4']] - data[idx['A1']]
    o1a2, o2a1 = data[idx['O1']] - data[idx['A2']], data[idx['O2']] - data[idx['A1']]
    
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    return mne.io.RawArray(final_data, info, verbose='ERROR'), edf_time

def process_subject(edf_path: str, txt_dir: str) -> Dict[str, np.ndarray]:
    """Executes the full preprocessing and QC slicing pipeline for one subject."""
    clean_raw, edf_time = clean_and_reconstruct(edf_path)
    max_time = clean_raw.times[-1]
    
    # Build QC Zones
    exclusion_zones = build_qc_masks(
        os.path.join(txt_dir, "REM.txt"),
        os.path.join(txt_dir, "Sleep Profile Reliability.txt"),
        edf_time, max_time
    )

    results = {"macro_onset": None, "micro_arousals": [], "micro_spindles": []}

    # Slicer A: Macro Onset
    prof_path = os.path.join(txt_dir, "Sleep Profile.txt")
    prof_offset = get_absolute_anchor(prof_path, edf_time)
    
    try:
        with open(prof_path, 'r') as f:
            prev_stage = None
            for line in f:
                if ';' not in line or 'Start' in line: continue
                parts = line.split(';')
                t_str, stage = parts[0].strip(), parts[1].strip()
                
                if prev_stage in ['Wake', 'Movement'] and stage == 'Stage 1':
                    onset_sec = parse_relative_sec(t_str) + prof_offset
                    start, end = onset_sec - 150, onset_sec + 150
                    if is_valid_slice(start, end, exclusion_zones) and start >= 0 and end <= max_time:
                        results["macro_onset"], _ = clean_raw.copy().crop(tmin=start, tmax=end).get_data(return_times=True)
                    break
                prev_stage = stage
    except FileNotFoundError: pass

    # Slicer B & C: Micro Transients (Arousals & Spindles)
    for event_file, dict_key in [("Arousal.txt", "micro_arousals"), ("Spindle K.txt", "micro_spindles")]:
        path = os.path.join(txt_dir, event_file)
        offset = get_absolute_anchor(path, edf_time)
        try:
            with open(path, 'r') as f:
                for line in f:
                    if '-' in line and ';' in line:
                        t_range = line.split(';')[0]
                        start = parse_relative_sec(t_range.split('-')[0]) + offset
                        end = parse_relative_sec(t_range.split('-')[1]) + offset
                        center = (start + end) / 2.0
                        
                        s_start, s_end = center - 1.5, center + 1.5 # 3-second window
                        if is_valid_slice(s_start, s_end, exclusion_zones) and s_start >= 0 and s_end <= max_time:
                            arr, _ = clean_raw.copy().crop(tmin=s_start, tmax=s_end).get_data(return_times=True)
                            results[dict_key].append(arr)
        except FileNotFoundError: pass

    results["micro_arousals"] = np.array(results["micro_arousals"]) if results["micro_arousals"] else np.empty(0)
    results["micro_spindles"] = np.array(results["micro_spindles"]) if results["micro_spindles"] else np.empty(0)
    
    print(f"Extraction Complete | Onset Array: {results['macro_onset'] is not None} | Valid Arousals: {len(results['micro_arousals'])} | Valid Spindles: {len(results['micro_spindles'])}")
    return results

# Ensure this is pushed all the way to the left margin (0 spaces)
if __name__ == "__main__":
    edf_file = "/kaggle/input/datasets/mridulmondal/normal01/Raw_Signal_Psycophysiological_Insomnia_01.edf"
    txt_folder = "/kaggle/input/datasets/johanliebert28/ensominia-psg-outputs/PSG_Output_Psycophysiological_Insomnia_01"
    
    # Must be indented exactly 4 spaces under the if statement
    out = process_subject(edf_file, txt_folder)
    
    print("Test extraction successful.")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


ImportError: The picard package is required to use method='picard', package was not found

In [5]:
!pip install mne numpy matplotlib python-picard

In [10]:
import mne
import numpy as np
import matplotlib.pyplot as plt

def run_phase_1_cleaning(edf_path: str, plot_validation: bool = True) -> mne.io.RawArray:
    """
    Executes Phase 1: Continuous Signal Cleaning and Channel Derivation.
    """
    print(f"--- INGESTING: {edf_path} ---")
    
    # ---------------------------------------------------------
    # STEP 1: BASE INGESTION & FILTERING
    # ---------------------------------------------------------
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    eog_chans = ['EOG1', 'EOG2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    # (Update Step 1) Isolate physical sensors using the modern MNE syntax
    raw_base = raw.copy().pick(base_eeg + eog_chans + emg_chans)
    
    # Apply 1 - 40 Hz Bandpass Filter (Zero-phase FIR)
    print("Applying 1-40 Hz Bandpass Filter...")
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    # Save a copy for the "Before" plot
    raw_before_ica = raw_base.copy()

    # ---------------------------------------------------------
    # STEP 2: EOG-ANCHORED ICA
    # ---------------------------------------------------------
    print("Computing EOG-Anchored ICA Matrix...")
    # Use 'picard' for highly robust sub-Gaussian artifact isolation
    ica = mne.preprocessing.ICA(n_components=10, method='picard', max_iter='auto', random_state=42)
    
     # (Update Step 2) Fit ICA using the modern syntax
    ica.fit(raw_base.copy().pick(base_eeg + eog_chans), verbose='ERROR')
    
    # CRITICAL FIX: Lower the threshold to force artifact detection
    # 'threshold=3.0' uses a 3 z-score cutoff instead of the strict default.
    # 'measure=correlation' forces it to strictly compare against EOG sensors.
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_base, 
        ch_name=eog_chans, 
        threshold=3.0, 
        measure='correlation',
        verbose='ERROR'
    )
    print(f"  -> Isolated Ocular ICA Components: {eog_indices}")
    
    # Zero out the artifacts and reconstruct
    ica.exclude = eog_indices
    raw_clean = ica.apply(raw_base, verbose='ERROR')
    
    # ---------------------------------------------------------
    # STEP 3: 17-CHANNEL RECONSTRUCTION
    # ---------------------------------------------------------
    print("Re-deriving Mastoid Referenced Channels...")
    data = raw_clean.get_data()
    idx = {ch: i for i, ch in enumerate(raw_clean.ch_names)}
    
    # Re-derive the mixed channels from the perfectly clean base channels
    c4a1 = data[idx['C4']] - data[idx['A1']]
    c3a2 = data[idx['C3']] - data[idx['A2']]
    f3a2 = data[idx['F3']] - data[idx['A2']]
    f4a1 = data[idx['F4']] - data[idx['A1']]
    o1a2 = data[idx['O1']] - data[idx['A2']]
    o2a1 = data[idx['O2']] - data[idx['A1']]
    
    # Stack the arrays into the final topological order
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch_names = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch_names, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    
    final_raw = mne.io.RawArray(final_data, info, verbose='ERROR')
    print(f"Phase 1 Complete. Output Tensor Shape: {final_raw.get_data().shape}\n")

    # ---------------------------------------------------------
    # STEP 4: VISUAL VALIDATION (Proof of Math)
    # ---------------------------------------------------------
    if plot_validation and len(eog_indices) > 0:
        print("Rendering Validation Plots. Close window to continue...")
        # Plot 10 seconds of Frontal channels (where blinks happen) to prove removal
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
        
        start_time, duration = 100, 10  # Seconds to plot
        start_idx, end_idx = int(start_time * sfreq), int((start_time + duration) * sfreq)
        time_axis = np.linspace(start_time, start_time + duration, end_idx - start_idx)
        
        # Plot Before ICA (F3)
        f3_before = raw_before_ica.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[0].plot(time_axis, f3_before, color='red', linewidth=1)
        axes[0].set_title("Before ICA: F3 Channel (Notice the massive eye blinks)")
        axes[0].grid(True, alpha=0.3)
        
        # Plot After ICA (F3)
        f3_after = raw_clean.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[1].plot(time_axis, f3_after, color='blue', linewidth=1)
        axes[1].set_title("After ICA: F3 Channel (Blinks removed, underlying Beta preserved)")
        axes[1].grid(True, alpha=0.3)
        
        plt.xlabel("Time (Seconds)")
        plt.tight_layout()
        plt.show()

    return final_raw

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # Point this to ANY of your .edf files to test the math
    test_file = "/kaggle/input/datasets/mridulmondal/normal01/Raw_Signal_Psycophysiological_Insomnia_01.edf" 
    
    # Run the cleaning phase and trigger the visualizer
    clean_tensor = run_phase_1_cleaning(test_file, plot_validation=True)

--- INGESTING: /kaggle/input/datasets/mridulmondal/normal01/Raw_Signal_Psycophysiological_Insomnia_01.edf ---
Applying 1-40 Hz Bandpass Filter...
Computing EOG-Anchored ICA Matrix...
  -> Isolated Ocular ICA Components: []
Re-deriving Mastoid Referenced Channels...
Phase 1 Complete. Output Tensor Shape: (17, 7368192)



In [9]:
import mne
import numpy as np
import matplotlib.pyplot as plt

def run_phase_1_cleaning(edf_path: str, plot_validation: bool = True) -> mne.io.RawArray:
    """
    Executes Phase 1: Continuous Signal Cleaning and Channel Derivation.
    Uses a Virtual EOG dipole to force ICA to capture sub-Gaussian ocular artifacts.
    
    Args:
        edf_path (str): Path to the EDF file.
        plot_validation (bool): Whether to plot before/after ICA validation.
        
    Returns:
        mne.io.RawArray: The cleaned, 17-channel reconstructed continuous tensor.
    """
    print(f"--- INGESTING: {edf_path} ---")
    
    # ---------------------------------------------------------
    # STEP 1: BASE INGESTION & FILTERING (VIRTUAL EOG FIX)
    # ---------------------------------------------------------
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    eog_chans = ['EOG1', 'EOG2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    # Isolate physical sensors using modern syntax
    raw_base = raw.copy().pick(base_eeg + eog_chans + emg_chans)
    
    # Force MNE to recognize EOGs and scale them correctly (µV correction)
    raw_base.set_channel_types({'EOG1': 'eog', 'EOG2': 'eog'})
    
    # Create Virtual Bipolar channel (EOG1 - EOG2) to maximize the blink dipole
    mne.set_bipolar_reference(
        raw_base, 
        anode=['EOG1'], 
        cathode=['EOG2'], 
        ch_name='Virtual_EOG', 
        drop_refs=False, 
        copy=False,
        verbose='ERROR'
    )
    raw_base.set_channel_types({'Virtual_EOG': 'eog'})
    
    print("Applying 1-40 Hz Bandpass Filter...")
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    # Save a copy for the "Before" plot
    raw_before_ica = raw_base.copy()

    # ---------------------------------------------------------
    # STEP 2: EOG-ANCHORED ICA
    # ---------------------------------------------------------
    print("Computing EOG-Anchored ICA Matrix...")
    ica = mne.preprocessing.ICA(n_components=8, method='picard', max_iter='auto', random_state=42)
    
    # Fit ICA on base EEG + original EOGs (Exclude EMG and Virtual_EOG from matrix)
    ica.fit(raw_base.copy().pick(base_eeg + eog_chans), verbose='ERROR')
    
    # Point correlation exclusively at the massive Virtual EOG dipole
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_base, 
        ch_name='Virtual_EOG', 
        threshold=2.5, 
        measure='correlation',
        verbose='ERROR'
    )
    print(f"  -> Isolated Ocular ICA Components: {eog_indices}")
    
    # Apply cleaning and drop the temporary Virtual_EOG
    ica.exclude = eog_indices
    raw_clean = ica.apply(raw_base, verbose='ERROR')
    raw_clean.drop_channels(['Virtual_EOG'])
    
    # ---------------------------------------------------------
    # STEP 3: 17-CHANNEL RECONSTRUCTION
    # ---------------------------------------------------------
    print("Re-deriving Mastoid Referenced Channels...")
    data = raw_clean.get_data()
    idx = {ch: i for i, ch in enumerate(raw_clean.ch_names)}
    
    # Re-derive mixed channels from the perfectly clean base channels
    c4a1 = data[idx['C4']] - data[idx['A1']]
    c3a2 = data[idx['C3']] - data[idx['A2']]
    f3a2 = data[idx['F3']] - data[idx['A2']]
    f4a1 = data[idx['F4']] - data[idx['A1']]
    o1a2 = data[idx['O1']] - data[idx['A2']]
    o2a1 = data[idx['O2']] - data[idx['A1']]
    
    # Stack arrays into the final topological order
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch_names = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch_names, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    
    final_raw = mne.io.RawArray(final_data, info, verbose='ERROR')
    print(f"Phase 1 Complete. Output Tensor Shape: {final_raw.get_data().shape}\n")

    # ---------------------------------------------------------
    # STEP 4: VISUAL VALIDATION (Proof of Math)
    # ---------------------------------------------------------
    if plot_validation and len(eog_indices) > 0:
        print("Rendering Validation Plots. Close window to continue...")
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
        
        start_time, duration = 100, 10  # Seconds to plot
        start_idx, end_idx = int(start_time * sfreq), int((start_time + duration) * sfreq)
        time_axis = np.linspace(start_time, start_time + duration, end_idx - start_idx)
        
        # Plot Before ICA (F3)
        f3_before = raw_before_ica.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[0].plot(time_axis, f3_before, color='red', linewidth=1)
        axes[0].set_title("Before ICA: F3 Channel (Notice the massive eye blinks)")
        axes[0].grid(True, alpha=0.3)
        
        # Plot After ICA (F3)
        f3_after = final_raw.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[1].plot(time_axis, f3_after, color='blue', linewidth=1)
        axes[1].set_title("After ICA: F3 Channel (Blinks removed, underlying Beta preserved)")
        axes[1].grid(True, alpha=0.3)
        
        plt.xlabel("Time (Seconds)")
        plt.tight_layout()
        plt.show()

    return final_raw

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # Ensure this path matches your Kaggle or local environment
    test_file = "/kaggle/input/datasets/mridulmondal/normal01/Normal_Subject_01.edf"
    
    # Execute the cleaning phase
    clean_tensor = run_phase_1_cleaning(test_file, plot_validation=True)

--- INGESTING: /kaggle/input/datasets/mridulmondal/normal01/Normal_Subject_01.edf ---
Applying 1-40 Hz Bandpass Filter...
Computing EOG-Anchored ICA Matrix...
  -> Isolated Ocular ICA Components: []
Re-deriving Mastoid Referenced Channels...
Phase 1 Complete. Output Tensor Shape: (17, 7366400)



In [12]:
import mne
import numpy as np
import matplotlib.pyplot as plt

def run_phase_1_cleaning(edf_path: str, plot_validation: bool = True) -> mne.io.RawArray:
    """
    Executes Phase 1: Continuous Signal Cleaning and Channel Derivation.
    Uses a Virtual EOG dipole to force ICA to capture sub-Gaussian ocular artifacts.
    
    Args:
        edf_path (str): Path to the EDF file.
        plot_validation (bool): Whether to plot before/after ICA validation.
        
    Returns:
        mne.io.RawArray: The cleaned, 17-channel reconstructed continuous tensor.
    """
    print(f"--- INGESTING: {edf_path} ---")
    
    # ---------------------------------------------------------
    # STEP 1: BASE INGESTION & FILTERING (VIRTUAL EOG FIX)
    # ---------------------------------------------------------
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    eog_chans = ['EOG1', 'EOG2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    # Isolate physical sensors using modern syntax
    raw_base = raw.copy().pick(base_eeg + eog_chans + emg_chans)
    
    # Force MNE to recognize EOGs and scale them correctly (µV correction)
    raw_base.set_channel_types({'EOG1': 'eog', 'EOG2': 'eog'})
    
    # Create Virtual Bipolar channel (EOG1 - EOG2) to maximize the blink dipole
    mne.set_bipolar_reference(
        raw_base, 
        anode=['EOG1'], 
        cathode=['EOG2'], 
        ch_name='Virtual_EOG', 
        drop_refs=False, 
        copy=False,
        verbose='ERROR'
    )
    raw_base.set_channel_types({'Virtual_EOG': 'eog'})
    
    print("Applying 1-40 Hz Bandpass Filter...")
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    # Save a copy for the "Before" plot
    raw_before_ica = raw_base.copy()

    # ---------------------------------------------------------
    # STEP 2: EOG-ANCHORED ICA
    # ---------------------------------------------------------
    print("Computing EOG-Anchored ICA Matrix...")
    ica = mne.preprocessing.ICA(n_components=8, method='picard', max_iter='auto', random_state=42)
    
    # Fit ICA on base EEG + original EOGs (Exclude EMG and Virtual_EOG from matrix)
    ica.fit(raw_base.copy().pick(base_eeg + eog_chans), verbose='ERROR')
    
    # Point correlation exclusively at the massive Virtual EOG dipole
    eog_indices, eog_scores = ica.find_bads_eog(
        raw_base, 
        ch_name='Virtual_EOG', 
        threshold=2.5, 
        measure='correlation',
        verbose='ERROR'
    )
    print(f"  -> Isolated Ocular ICA Components: {eog_indices}")
    
    # Apply cleaning and drop the temporary Virtual_EOG
    ica.exclude = eog_indices
    raw_clean = ica.apply(raw_base, verbose='ERROR')
    raw_clean.drop_channels(['Virtual_EOG'])
    
    # ---------------------------------------------------------
    # STEP 3: 17-CHANNEL RECONSTRUCTION
    # ---------------------------------------------------------
    print("Re-deriving Mastoid Referenced Channels...")
    data = raw_clean.get_data()
    idx = {ch: i for i, ch in enumerate(raw_clean.ch_names)}
    
    # Re-derive mixed channels from the perfectly clean base channels
    c4a1 = data[idx['C4']] - data[idx['A1']]
    c3a2 = data[idx['C3']] - data[idx['A2']]
    f3a2 = data[idx['F3']] - data[idx['A2']]
    f4a1 = data[idx['F4']] - data[idx['A1']]
    o1a2 = data[idx['O1']] - data[idx['A2']]
    o2a1 = data[idx['O2']] - data[idx['A1']]
    
    # Stack arrays into the final topological order
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch_names = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch_names, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    
    final_raw = mne.io.RawArray(final_data, info, verbose='ERROR')
    print(f"Phase 1 Complete. Output Tensor Shape: {final_raw.get_data().shape}\n")

    # ---------------------------------------------------------
    # STEP 4: VISUAL VALIDATION (Proof of Math)
    # ---------------------------------------------------------
    if plot_validation and len(eog_indices) > 0:
        print("Rendering Validation Plots. Close window to continue...")
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
        
        start_time, duration = 100, 10  # Seconds to plot
        start_idx, end_idx = int(start_time * sfreq), int((start_time + duration) * sfreq)
        time_axis = np.linspace(start_time, start_time + duration, end_idx - start_idx)
        
        # Plot Before ICA (F3)
        f3_before = raw_before_ica.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[0].plot(time_axis, f3_before, color='red', linewidth=1)
        axes[0].set_title("Before ICA: F3 Channel (Notice the massive eye blinks)")
        axes[0].grid(True, alpha=0.3)
        
        # Plot After ICA (F3)
        f3_after = final_raw.get_data(picks=['F3'])[0, start_idx:end_idx]
        axes[1].plot(time_axis, f3_after, color='blue', linewidth=1)
        axes[1].set_title("After ICA: F3 Channel (Blinks removed, underlying Beta preserved)")
        axes[1].grid(True, alpha=0.3)
        
        plt.xlabel("Time (Seconds)")
        plt.tight_layout()
        plt.show()

    return final_raw

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # Ensure this path matches your Kaggle or local environment
    test_file = "/kaggle/input/datasets/mridulmondal/normal01/Raw_Signal_Psycophysiological_Insomnia_01.edf"
    
    # Execute the cleaning phase
    clean_tensor = run_phase_1_cleaning(test_file, plot_validation=True)

--- INGESTING: /kaggle/input/datasets/mridulmondal/normal01/Raw_Signal_Psycophysiological_Insomnia_01.edf ---
Applying 1-40 Hz Bandpass Filter...
Computing EOG-Anchored ICA Matrix...
  -> Isolated Ocular ICA Components: []
Re-deriving Mastoid Referenced Channels...
Phase 1 Complete. Output Tensor Shape: (17, 7368192)



In [2]:
import os
import mne
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

# ==========================================
# 1. PHASE 1: CONTINUOUS TENSOR INGESTION (NO ICA)
# ==========================================
def run_phase_1_baseline(edf_path: str) -> Tuple[mne.io.RawArray, datetime]:
    """Ingests EDF, applies 1-40Hz filter, and builds 17-node spatial graph."""
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    edf_time = raw.info['meas_date'].replace(tzinfo=None) 
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    # Drop EOG, keep EEG + EMG
    raw_base = raw.copy().pick(base_eeg + emg_chans)
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    data = raw_base.get_data()
    idx = {ch: i for i, ch in enumerate(raw_base.ch_names)}
    
    # Re-derive Mastoids
    c4a1, c3a2 = data[idx['C4']] - data[idx['A1']], data[idx['C3']] - data[idx['A2']]
    f3a2, f4a1 = data[idx['F3']] - data[idx['A2']], data[idx['F4']] - data[idx['A1']]
    o1a2, o2a1 = data[idx['O1']] - data[idx['A2']], data[idx['O2']] - data[idx['A1']]
    
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch_names = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch_names, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    
    return mne.io.RawArray(final_data, info, verbose='ERROR'), edf_time

# ==========================================
# 2. TEMPORAL MATH & QC SHIELDS
# ==========================================
def parse_txt_start_time(filepath: str) -> Optional[datetime]:
    try:
        with open(filepath, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if i > 10: break
                if "Start Time:" in line:
                    t_str = line.split("Start Time:")[1].strip()
                    return datetime.strptime(t_str, "%m/%d/%Y %I:%M:%S %p")
    except Exception: pass
    return None

def compute_elapsed_seconds(event_time_str: str, txt_start_time: datetime, edf_start_time: datetime) -> float:
    t_str = event_time_str.replace(',', '.') 
    event_t = datetime.strptime(t_str.strip(), "%H:%M:%S.%f").time()
    event_dt = datetime.combine(txt_start_time.date(), event_t)
    if event_t.hour < (txt_start_time.hour - 4):  
        event_dt += timedelta(days=1)
    return (event_dt - edf_start_time).total_seconds()

def build_qc_masks(txt_dir: str, edf_time: datetime, max_time: float) -> List[Tuple[float, float]]:
    exclusion_zones = []
    
    # Matches exactly to your provided folder structure
    rem_path = os.path.join(txt_dir, "REM.txt")
    rem_start = parse_txt_start_time(rem_path)
    if rem_start and os.path.exists(rem_path):
        with open(rem_path, 'r') as f:
            for line in f:
                if '-' in line and 'REM' in line:
                    t_range = line.split(';')[0]
                    start = compute_elapsed_seconds(t_range.split('-')[0], rem_start, edf_time)
                    end = compute_elapsed_seconds(t_range.split('-')[1], rem_start, edf_time)
                    exclusion_zones.append((start, end))

    # Updated to match "Sleep Profile Reliability.txt"
    rel_path = os.path.join(txt_dir, "Sleep Profile Reliability.txt") 
    rel_start = parse_txt_start_time(rel_path)
    if rel_start and os.path.exists(rel_path):
        with open(rel_path, 'r') as f:
            for line in f:
                if '<35% Reliability' in line:
                    t_str = line.split(';')[0]
                    start = compute_elapsed_seconds(t_str, rel_start, edf_time)
                    exclusion_zones.append((start, start + 30.0)) 
                    
    return exclusion_zones

def is_valid_slice(start: float, end: float, exclusion_zones: List[Tuple[float, float]]) -> bool:
    for ex_start, ex_end in exclusion_zones:
        if max(start, ex_start) < min(end, ex_end):
            return False
    return True

# ==========================================
# 3. STAGE ROUTER & DISK WRITER
# ==========================================
def extract_and_save_all_stages(
    raw: mne.io.RawArray, 
    edf_time: datetime, 
    txt_dir: str, 
    patient_id: str, 
    output_dir: str,
    epoch_duration: float = 30.0
) -> None:
    sfreq = raw.info['sfreq']
    samples_per_epoch = int(epoch_duration * sfreq)
    max_time = raw.times[-1]
    
    exclusion_zones = build_qc_masks(txt_dir, edf_time, max_time)
    
    # Matches exactly to your provided folder structure
    prof_path = os.path.join(txt_dir, "Sleep Profile.txt")
    prof_start = parse_txt_start_time(prof_path)
    if not prof_start:
        raise ValueError(f"CRITICAL: Could not parse Start Time from {prof_path}")

    target_stages = ['Wake', 'Stage 1', 'Stage 2', 'Stage 3']
    stage_buffers: Dict[str, List[np.ndarray]] = {stage: [] for stage in target_stages}
    
    with open(prof_path, 'r') as f:
        for line in f:
            if ';' not in line or 'Start Time' in line or 'Events list' in line: 
                continue
                
            parts = line.split(';')
            t_str, stage = parts[0].strip(), parts[1].strip()
            
            if stage not in target_stages:
                continue
                
            start_sec = compute_elapsed_seconds(t_str, prof_start, edf_time)
            end_sec = start_sec + epoch_duration
            
            if is_valid_slice(start_sec, end_sec, exclusion_zones) and start_sec >= 0 and end_sec <= max_time:
                epoch_data, _ = raw.copy().crop(tmin=start_sec, tmax=end_sec).get_data(return_times=True)
                if epoch_data.shape[1] == samples_per_epoch:
                    stage_buffers[stage].append(epoch_data)

    os.makedirs(output_dir, exist_ok=True)
    print(f"\n--- Extraction Results for {patient_id} ---")
    
    for stage in target_stages:
        if not stage_buffers[stage]:
            print(f"{stage}: 0 valid epochs found. Skipping save.")
            continue
            
        stage_tensor = np.stack(stage_buffers[stage], axis=0).astype(np.float32)
        safe_stage_name = stage.replace(" ", "_")
        file_path = os.path.join(output_dir, f"{patient_id}_{safe_stage_name}.npy")
        
        np.save(file_path, stage_tensor)
        print(f"{stage}: Saved Tensor Shape {stage_tensor.shape} -> {file_path}")

# ==========================================
# EXECUTION WORKFLOW
# ==========================================
if __name__ == "__main__":
    # 1. Define your strict Kaggle paths based on your layout
    KAGGLE_BASE = "/kaggle/input/datasets/mridulmondal" # Adjust base path if needed
    KAGGLE_BASE1 = "/kaggle/input/datasets/johanliebert28"
    edf_file = os.path.join(KAGGLE_BASE, "normal01", "Normal_Subject_01.edf")
    txt_directory = os.path.join(KAGGLE_BASE1, "Ensominia_PSG_Outputs", "PSG_Output_Normal_Subject_01")
    output_directory = "./processed_epochs" # Saves to Kaggle working directory
    patient_id = "Normal_Subject_01"
    
    # 2. Execute Pipeline
    try:
        clean_tensor, hardware_start = run_phase_1_baseline(edf_file)
        extract_and_save_all_stages(
            raw=clean_tensor, 
            edf_time=hardware_start, 
            txt_dir=txt_directory, 
            patient_id=patient_id, 
            output_dir=output_directory
        )
    except Exception as e:
        print(f"Pipeline Failed: {str(e)}")

Pipeline Failed: CRITICAL: Could not parse Start Time from /kaggle/input/datasets/johanliebert28/Ensominia_PSG_Outputs/PSG_Output_Normal_Subject_01/Sleep Profile.txt


In [5]:
import os
from datetime import datetime

def debug_parse_time(filepath: str) -> None:
    print(f"--- Debugging File: {filepath} ---")
    
    if not os.path.exists(filepath):
        print("CRITICAL ERROR: File does not exist at this path. Check your directory spelling.")
        return
        
    try:
        # Using utf-8 fallback if latin-1 fails
        with open(filepath, 'r', encoding='latin-1') as f:
            found = False
            for i, line in enumerate(f):
                if i > 20: # Increased search depth
                    break
                    
                if "Start Time:" in line:
                    found = True
                    print(f"Found on line {i}: '{line.strip()}'")
                    t_str = line.split("Start Time:")[1].strip()
                    print(f"Extracted String: '{t_str}'")
                    
                    try:
                        # Attempt to parse with the original strict format
                        dt = datetime.strptime(t_str, "%m/%d/%Y %I:%M:%S %p")
                        print(f"Success! Parsed as: {dt}")
                    except ValueError as ve:
                        print(f"DATETIME VALUE ERROR: {ve}")
                        print("The format in this file does not match '%m/%d/%Y %I:%M:%S %p'.")
                    break
                    
            if not found:
                print("ERROR: 'Start Time:' string was never found in the first 20 lines.")
                
    except Exception as e:
        print(f"FILE ACCESS/ENCODING ERROR: {e}")

# EXECUTION
# EXECUTION
# Notice the explicit addition of the file name and the updated lowercase base path.
test_path = "/kaggle/input/datasets/johanliebert28/ensominia-psg-outputs/PSG_Output_Normal_Subject_01/Sleep Profile.txt"

debug_parse_time(test_path)

--- Debugging File: /kaggle/input/datasets/johanliebert28/ensominia-psg-outputs/PSG_Output_Normal_Subject_01/Sleep Profile.txt ---
Found on line 1: 'Start Time: 1/26/2015 10:56:00 PM'
Extracted String: '1/26/2015 10:56:00 PM'
Success! Parsed as: 2015-01-26 22:56:00


In [13]:
import os
import mne
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

# ==========================================
# 1. PHASE 1: CONTINUOUS TENSOR INGESTION
# ==========================================
def run_phase_1_baseline(edf_path: str) -> Tuple[mne.io.RawArray, datetime]:
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose='ERROR')
    sfreq = raw.info['sfreq']
    edf_time = raw.info['meas_date'].replace(tzinfo=None) 
    
    base_eeg = ['F3', 'F4', 'C3', 'C4', 'O1', 'O2', 'A1', 'A2']
    emg_chans = ['EMG', 'EMG1', 'EMG2']
    
    raw_base = raw.copy().pick(base_eeg + emg_chans)
    raw_base.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin', verbose='ERROR')
    
    data = raw_base.get_data()
    idx = {ch: i for i, ch in enumerate(raw_base.ch_names)}
    
    c4a1, c3a2 = data[idx['C4']] - data[idx['A1']], data[idx['C3']] - data[idx['A2']]
    f3a2, f4a1 = data[idx['F3']] - data[idx['A2']], data[idx['F4']] - data[idx['A1']]
    o1a2, o2a1 = data[idx['O1']] - data[idx['A2']], data[idx['O2']] - data[idx['A1']]
    
    final_data = np.vstack([
        data[idx['F3']], data[idx['F4']], data[idx['C3']], data[idx['C4']], 
        data[idx['O1']], data[idx['O2']], data[idx['A1']], data[idx['A2']],
        c4a1, c3a2, f3a2, f4a1, o1a2, o2a1,
        data[idx['EMG']], data[idx['EMG1']], data[idx['EMG2']]
    ])
    
    final_ch_names = base_eeg + ['C4A1', 'C3A2', 'F3A2', 'F4A1', 'O1A2', 'O2A1'] + emg_chans
    info = mne.create_info(ch_names=final_ch_names, sfreq=sfreq, ch_types=['eeg']*14 + ['emg']*3)
    
    return mne.io.RawArray(final_data, info, verbose='ERROR'), edf_time

# ==========================================
# 2. TEMPORAL MATH & QC SHIELDS
# ==========================================
def parse_txt_start_time(filepath: str) -> Optional[datetime]:
    if not os.path.exists(filepath):
        print(f"Warning: File not found at {filepath}.")
        return None

    try:
        # latin-1 handles the 'µV' byte safely
        with open(filepath, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if i > 20: break 
                if "Start Time:" in line:
                    t_str = line.split("Start Time:")[1].strip()
                    return datetime.strptime(t_str, "%m/%d/%Y %I:%M:%S %p")
    except Exception as e:
        print(f"File Access Error on {filepath}: {e}")
    return None

def compute_elapsed_seconds(event_time_str: str, txt_start_time: datetime, edf_start_time: datetime) -> float:
    t_str = event_time_str.replace(',', '.') 
    event_t = datetime.strptime(t_str.strip(), "%H:%M:%S.%f").time()
    event_dt = datetime.combine(txt_start_time.date(), event_t)
    if event_t.hour < (txt_start_time.hour - 4):  
        event_dt += timedelta(days=1)
    return (event_dt - edf_start_time).total_seconds()

def build_qc_masks(txt_dir: str, edf_time: datetime, max_time: float) -> List[Tuple[float, float]]:
    exclusion_zones = []
    
    rem_path = os.path.join(txt_dir, "REM.txt")
    rem_start = parse_txt_start_time(rem_path)
    if rem_start:
        with open(rem_path, 'r', encoding='latin-1') as f:
            for line in f:
                # Ensure it's a valid data line
                if ';' not in line or 'Events list' in line:
                    continue
                if '-' in line and 'REM' in line:
                    t_range = line.split(';')[0]
                    start = compute_elapsed_seconds(t_range.split('-')[0], rem_start, edf_time)
                    end = compute_elapsed_seconds(t_range.split('-')[1], rem_start, edf_time)
                    exclusion_zones.append((start, end))

    rel_path = os.path.join(txt_dir, "Sleep Profile Reliability.txt") 
    rel_start = parse_txt_start_time(rel_path)
    if rel_start:
        with open(rel_path, 'r', encoding='latin-1') as f:
            for line in f:
                # CRITICAL FIX: Skip headers and malformed lines
                if ';' not in line or 'Events list' in line:
                    continue
                if '<35% Reliability' in line:
                    t_str = line.split(';')[0]
                    start = compute_elapsed_seconds(t_str, rel_start, edf_time)
                    exclusion_zones.append((start, start + 30.0)) 
                    
    return exclusion_zones

# ==========================================
# 3. STAGE ROUTER & DISK WRITER
# ==========================================
def extract_and_save_all_stages(
    raw: mne.io.RawArray, 
    edf_time: datetime, 
    txt_dir: str, 
    patient_id: str, 
    output_dir: str,
    epoch_duration: float = 30.0
) -> None:
    sfreq = raw.info['sfreq']
    samples_per_epoch = int(epoch_duration * sfreq)
    max_time = raw.times[-1]
    
    exclusion_zones = build_qc_masks(txt_dir, edf_time, max_time)
    
    prof_path = os.path.join(txt_dir, "Sleep Profile.txt")
    prof_start = parse_txt_start_time(prof_path)
    if not prof_start:
        raise ValueError(f"FATAL: Hypnogram missing or unparseable at {prof_path}")

    target_stages = ['Wake', 'Stage 1', 'Stage 2', 'Stage 3']
    stage_buffers: Dict[str, List[np.ndarray]] = {stage: [] for stage in target_stages}
    
    with open(prof_path, 'r', encoding='latin-1') as f:
        for line in f:
            if ';' not in line or 'Start Time' in line or 'Events list' in line: 
                continue
                
            parts = line.split(';')
            t_str, stage = parts[0].strip(), parts[1].strip()
            
            if stage not in target_stages:
                continue
                
            start_sec = compute_elapsed_seconds(t_str, prof_start, edf_time)
            end_sec = start_sec + epoch_duration
            
            # Ensure the slice falls within the EDF physical hardware bounds
            if is_valid_slice(start_sec, end_sec, exclusion_zones) and start_sec >= 0 and end_sec <= max_time:
                epoch_data, _ = raw.copy().crop(tmin=start_sec, tmax=end_sec).get_data(return_times=True)
                
                # CRITICAL FIX: Truncate the MNE inclusive +1 sample to enforce strict (17, 7680) shape
                epoch_data = epoch_data[:, :samples_per_epoch]
                
                if epoch_data.shape[1] == samples_per_epoch:
                    stage_buffers[stage].append(epoch_data)

    os.makedirs(output_dir, exist_ok=True)
    print(f"\n--- Extraction Results for {patient_id} ---")
    
    for stage in target_stages:
        if not stage_buffers[stage]:
            print(f"{stage}: 0 valid epochs found. Skipping save.")
            continue
            
        stage_tensor = np.stack(stage_buffers[stage], axis=0).astype(np.float32)
        safe_stage_name = stage.replace(" ", "_")
        file_path = os.path.join(output_dir, f"{patient_id}_{safe_stage_name}.npy")
        
        np.save(file_path, stage_tensor)
        print(f"{stage}: Saved Tensor Shape {stage_tensor.shape} -> {file_path}")
# ==========================================
# EXECUTION WORKFLOW
# ==========================================
if __name__ == "__main__":
    edf_file = "/kaggle/input/datasets/mridulmondal/normal01/Normal_Subject_01.edf"
    txt_directory = "/kaggle/input/datasets/johanliebert28/ensominia-psg-outputs/PSG_Output_Normal_Subject_01"
    
    output_directory = "./processed_epochs"
    patient_id = "Normal_Subject_01"
    
    try:
        clean_tensor, hardware_start = run_phase_1_baseline(edf_file)
        extract_and_save_all_stages(
            raw=clean_tensor, 
            edf_time=hardware_start, 
            txt_dir=txt_directory, 
            patient_id=patient_id, 
            output_dir=output_directory
        )
    except Exception as e:
        import traceback
        print(f"Pipeline Failed: {str(e)}")
        print(traceback.format_exc())


--- Extraction Results for Normal_Subject_01 ---
Wake: Saved Tensor Shape (25, 17, 7680) -> ./processed_epochs/Normal_Subject_01_Wake.npy
Stage 1: Saved Tensor Shape (107, 17, 7680) -> ./processed_epochs/Normal_Subject_01_Stage_1.npy
Stage 2: Saved Tensor Shape (448, 17, 7680) -> ./processed_epochs/Normal_Subject_01_Stage_2.npy
Stage 3: Saved Tensor Shape (229, 17, 7680) -> ./processed_epochs/Normal_Subject_01_Stage_3.npy


In [12]:
import mne
import os
from datetime import datetime

# 1. Define Paths
edf_file = "/kaggle/input/datasets/mridulmondal/normal01/Normal_Subject_01.edf"
txt_path = "/kaggle/input/datasets/johanliebert28/ensominia-psg-outputs/PSG_Output_Normal_Subject_01/Sleep Profile.txt"

# 2. Extract EDF Hardware Clock & Duration
try:
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose='ERROR')
    edf_time = raw.info['meas_date'].replace(tzinfo=None)
    max_time = raw.times[-1]
    print(f"--- TEMPORAL DIAGNOSTIC ---")
    print(f"1. EDF Hardware Clock:  {edf_time}")
    print(f"2. EDF Total Duration:  {max_time} seconds ({max_time/3600:.2f} hours)")
except Exception as e:
    print(f"EDF Read Error: {e}")

# 3. Extract TXT Clinical Clock
try:
    txt_time = None
    with open(txt_path, 'r', encoding='latin-1') as f:
        for i, line in enumerate(f):
            if i > 20: break
            if "Start Time:" in line:
                t_str = line.split("Start Time:")[1].strip()
                txt_time = datetime.strptime(t_str, "%m/%d/%Y %I:%M:%S %p")
                print(f"3. TXT Clinical Clock:  {txt_time}")
                break
                
    # 4. Compute the Fatal Drift
    if txt_time and edf_time:
        drift = (txt_time - edf_time).total_seconds()
        print(f"\n--- THE DRIFT ---")
        print(f"Calculated Offset:      {drift} seconds")
        if drift < 0 or drift > max_time:
            print("CRITICAL FAILURE: The TXT clock is completely outside the EDF recording bounds.")
            print("This is why 0 epochs were extracted.")
except Exception as e:
    print(f"TXT Read Error: {e}")

--- TEMPORAL DIAGNOSTIC ---
1. EDF Hardware Clock:  2015-01-26 22:56:24
2. EDF Total Duration:  28774.99609375 seconds (7.99 hours)
3. TXT Clinical Clock:  2015-01-26 22:56:00

--- THE DRIFT ---
Calculated Offset:      -24.0 seconds
CRITICAL FAILURE: The TXT clock is completely outside the EDF recording bounds.
This is why 0 epochs were extracted.
